# Notebook 3: Data Wrangling & Target Variable Construction
## IBM Data Science Professional Certificate Capstone Project

### Learning Objectives:
1. Inspect the raw dataset using `head()`, `info()`, and `describe()`.
2. Identify missing values and apply appropriate imputation techniques.
3. Precisely define first stage landing success vs launch success.
4. Construct the binary target variable `Class` (1 for success, 0 for failure).
5. Prepare one-hot encoded feature matrix for machine learning models.

---
### 1. Data Cleaning Overview
Data wrangling ensures our dataset contains clean, consistent values without invalid null entries that would break machine learning algorithms.

In [ ]:
# Step 1: Import required libraries and load raw data
import pandas as pd
import numpy as np
import os

df = pd.read_csv("../data/raw/dataset_part_1.csv")
print(f"Loaded raw dataset with {df.shape[0]} rows and {df.shape[1]} columns.")
df.head(5)

### 2. Identifying and Handling Missing Values

In [ ]:
# Step 2: Check missing values across all columns
missing = df.isnull().sum()
print("Missing values per column:")
print(missing[missing > 0])

# Impute missing PayloadMass with the column mean
if "PayloadMass" in df.columns and df["PayloadMass"].isnull().sum() > 0:
    mean_val = df["PayloadMass"].mean()
    df["PayloadMass"] = df["PayloadMass"].fillna(mean_val)
    print(f"\nImputed missing PayloadMass values with dataset mean: {mean_val:.2f} kg")

### 3. Understanding Landing Outcomes & Constructing the Target Variable `Class`
In commercial rocketry, **launch success** (payload reaching orbit) is distinct from **first stage recovery success** (booster landing intact).
Let's inspect all unique `Outcome` values:

In [ ]:
# Step 3: Breakdown of raw landing outcomes
print("Distribution of landing outcomes:")
print(df["Outcome"].value_counts())

# Mapping outcomes to binary target:
# 1 = Successful Landing (True Ocean, True RTLS, True ASDS)
# 0 = Unsuccessful / Expended (False Ocean, False RTLS, False ASDS, None None, None ASDS)

landing_class = []
for outcome in df["Outcome"]:
    if isinstance(outcome, str) and outcome.startswith("True"):
        landing_class.append(1)
    else:
        landing_class.append(0)

df["Class"] = landing_class
df["Date"] = pd.to_datetime(df["Date"])
df["Year"] = df["Date"].dt.year

print(f"\nTarget variable 'Class' successfully created.")
print(f"Total Successful Landings (Class 1): {df['Class'].sum()} ({df['Class'].mean():.1%})")
print(f"Total Unsuccessful Attempts (Class 0): {(1 - df['Class']).sum()} ({(1 - df['Class'].mean()):.1%})")

In [ ]:
# Step 4: Save cleaned dataset_part_2.csv
os.makedirs("../data/processed", exist_ok=True)
part2_path = "../data/processed/dataset_part_2.csv"
df.to_csv(part2_path, index=False)
print(f"Cleaned dataset saved to: {part2_path}")

### 4. Feature Selection & One-Hot Encoding for Machine Learning
Categorical features like `Orbit`, `LaunchSite`, `LandingPad`, and `Serial` must be converted to numeric dummy variables.

In [ ]:
# Step 5: Create dummy variables using pd.get_dummies
features = df[[
    "FlightNumber", "PayloadMass", "Orbit", "LaunchSite",
    "Flights", "GridFins", "Reused", "Legs", "LandingPad", "Block", "ReusedCount", "Serial"
]].copy()

# Cast boolean flags to integers
features["GridFins"] = features["GridFins"].astype(int)
features["Reused"] = features["Reused"].astype(int)
features["Legs"] = features["Legs"].astype(int)

features_one_hot = pd.get_dummies(features, columns=["Orbit", "LaunchSite", "LandingPad", "Serial"], drop_first=False).astype(float)
df_part_3 = pd.concat([features_one_hot, df["Class"]], axis=1)

part3_path = "../data/processed/dataset_part_3.csv"
df_part_3.to_csv(part3_path, index=False)
print(f"One-hot encoded dataset saved to: {part3_path}")
print(f"Total ML features ready: {features_one_hot.shape[1]}")

---
### Practice Exercise & Check for Understanding:
1. Why do we encode categorical columns like `LaunchSite` into dummy variables before passing to machine learning models? *(Answer: Most ML algorithms require purely numeric matrices for mathematical distance and linear algebraic operations).*
2. What proportion of historical launches achieved successful first stage landing? *(Answer: 66.7%).*

### Summary & Next Steps:
We have prepared both the clean tabular dataset (`dataset_part_2.csv`) and the encoded machine learning matrix (`dataset_part_3.csv`). Next, in Notebook 4, we perform Exploratory Data Analysis with rich visualizations!